# Feature Engineering

## Imports

In [1]:
import pandas as pd

## Importar data

In [2]:
CLEAN_STORE_ROUTE = r"../data/clean/store_clean.csv"
CLEAN_TRAIN_ROUTE = r"../data/clean/train_clean.csv"

In [4]:
df_store = pd.read_csv(CLEAN_STORE_ROUTE)
df_train = pd.read_csv(CLEAN_TRAIN_ROUTE, dtype={"StateHoliday": str}) # Esto solo porque pandas vuelve a leer el csv e "infiere datos"

## Merge

In [5]:
df = df_train.merge(df_store, on="Store", how="left")

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Store                      1017209 non-null  int64  
 1   DayOfWeek                  1017209 non-null  int64  
 2   Date                       1017209 non-null  str    
 3   Sales                      1017209 non-null  int64  
 4   Customers                  1017209 non-null  int64  
 5   Open                       1017209 non-null  int64  
 6   Promo                      1017209 non-null  int64  
 7   StateHoliday               1017209 non-null  str    
 8   SchoolHoliday              1017209 non-null  int64  
 9   StoreType                  1014567 non-null  str    
 10  Assortment                 1014567 non-null  str    
 11  CompetitionDistance        1014567 non-null  float64
 12  CompetitionOpenSinceMonth  693861 non-null   float64
 13  CompetitionOpenSinceYea

In [7]:
df["StateHoliday"].unique()

<StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: str

## Preprocessing

### Nulos por store's en train pero no en store

Aparecieron nulos, esto porque hay tiendas que existen en train, pero no en store, eliminar tales registros

In [8]:
df = df.dropna(subset="StoreType")

In [9]:
df.info()

<class 'pandas.DataFrame'>
Index: 1014567 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Store                      1014567 non-null  int64  
 1   DayOfWeek                  1014567 non-null  int64  
 2   Date                       1014567 non-null  str    
 3   Sales                      1014567 non-null  int64  
 4   Customers                  1014567 non-null  int64  
 5   Open                       1014567 non-null  int64  
 6   Promo                      1014567 non-null  int64  
 7   StateHoliday               1014567 non-null  str    
 8   SchoolHoliday              1014567 non-null  int64  
 9   StoreType                  1014567 non-null  str    
 10  Assortment                 1014567 non-null  str    
 11  CompetitionDistance        1014567 non-null  float64
 12  CompetitionOpenSinceMonth  693861 non-null   float64
 13  CompetitionOpenSinceYear   6

### Conversión de feature "Date" a datetime

In [10]:
df["Date"] = pd.to_datetime(df["Date"])

In [11]:
df.info()

<class 'pandas.DataFrame'>
Index: 1014567 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1014567 non-null  int64         
 1   DayOfWeek                  1014567 non-null  int64         
 2   Date                       1014567 non-null  datetime64[us]
 3   Sales                      1014567 non-null  int64         
 4   Customers                  1014567 non-null  int64         
 5   Open                       1014567 non-null  int64         
 6   Promo                      1014567 non-null  int64         
 7   StateHoliday               1014567 non-null  str           
 8   SchoolHoliday              1014567 non-null  int64         
 9   StoreType                  1014567 non-null  str           
 10  Assortment                 1014567 non-null  str           
 11  CompetitionDistance        1014567 non-null  float64 

In [12]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0.0,0.0,0.0,NoPromo
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1.0,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1.0,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0.0,0.0,0.0,NoPromo
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0.0,0.0,0.0,NoPromo


### Nulos en CompetitionOpenSinceMonth/Year

Ahora faltan los registros de CompetitionOpenSinceMonth/Year

Para ello definiremos hasta cuando consideraremos parte de entrenamiento

Primero ordenamos las fechas

In [13]:
df = df.sort_values("Date")

Ahora vemos desde donde parte hasta donde llega

In [14]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
1017208,1115,2,2013-01-01,0,0,0,0,a,1,d,c,5350.0,NaN,NaN,1.0,22.0,2012.0,"Mar,Jun,Sept,Dec"
1016473,379,2,2013-01-01,0,0,0,0,a,1,d,a,6630.0,NaN,NaN,0.0,0.0,0.0,NoPromo
1016472,378,2,2013-01-01,0,0,0,0,a,1,a,c,2140.0,8.0,2012.0,0.0,0.0,0.0,NoPromo
1016471,377,2,2013-01-01,0,0,0,0,a,1,a,c,100.0,6.0,2010.0,1.0,18.0,2010.0,"Feb,May,Aug,Nov"
1016470,376,2,2013-01-01,0,0,0,0,a,1,a,a,160.0,8.0,2012.0,0.0,0.0,0.0,NoPromo


In [15]:
df.tail()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
745,746,5,2015-07-31,9082,638,1,1,0,1,d,c,4330.0,2.0,2011.0,1.0,35.0,2011.0,"Mar,Jun,Sept,Dec"
746,747,5,2015-07-31,10708,826,1,1,0,1,c,c,45740.0,8.0,2008.0,0.0,0.0,0.0,NoPromo
747,748,5,2015-07-31,7481,578,1,1,0,1,d,a,2380.0,3.0,2010.0,1.0,14.0,2011.0,"Jan,Apr,Jul,Oct"
741,742,5,2015-07-31,10460,1016,1,1,0,1,d,c,4380.0,NaN,NaN,0.0,0.0,0.0,NoPromo
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0.0,0.0,0.0,NoPromo


Llega hasta mas o menos mitades del 2015, asi que ocuparemos todo 2013 y todo 2014 como entrenamiento

Por lo que creamos un train temporal para calcular su mediana en CompetitionOpenSinceMonth y mediana en CompetitionOpenSinceYear

In [16]:
temporal_df_train = df[
    (df["Date"] < "2015-01-01")
]

In [17]:
median_competition_open_since_month = temporal_df_train["CompetitionOpenSinceMonth"].median()
median_competition_open_since_year = temporal_df_train["CompetitionOpenSinceYear"].median()

In [18]:
df["CompetitionOpenSinceMonth"] = df["CompetitionOpenSinceMonth"].fillna(median_competition_open_since_month)
df["CompetitionOpenSinceYear"] = df["CompetitionOpenSinceYear"].fillna(median_competition_open_since_year)

In [19]:
df.info()

<class 'pandas.DataFrame'>
Index: 1014567 entries, 1017208 to 0
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1014567 non-null  int64         
 1   DayOfWeek                  1014567 non-null  int64         
 2   Date                       1014567 non-null  datetime64[us]
 3   Sales                      1014567 non-null  int64         
 4   Customers                  1014567 non-null  int64         
 5   Open                       1014567 non-null  int64         
 6   Promo                      1014567 non-null  int64         
 7   StateHoliday               1014567 non-null  str           
 8   SchoolHoliday              1014567 non-null  int64         
 9   StoreType                  1014567 non-null  str           
 10  Assortment                 1014567 non-null  str           
 11  CompetitionDistance        1014567 non-null  float64 

## Reset index

Esto es necesario para que quedemos con un dataframe ordenado por fecha con indices ordenados

In [20]:
df = df.reset_index(drop=True)

## Feature engineering

### Crear features de tiempo con Date

Esto es necesario ya que los modelos no pueden interpretar una fecha

In [21]:
df["Day"] = df["Date"].dt.day

In [22]:
df["Month"] = df["Date"].dt.month

In [23]:
df["Year"] = df["Date"].dt.year

In [24]:
df["Week"] = df["Date"].dt.isocalendar().week

In [25]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Day,Month,Year,Week
0,1115,2,2013-01-01,0,0,0,0,a,1,d,...,8.0,2010.0,1.0,22.0,2012.0,"Mar,Jun,Sept,Dec",1,1,2013,1
1,379,2,2013-01-01,0,0,0,0,a,1,d,...,8.0,2010.0,0.0,0.0,0.0,NoPromo,1,1,2013,1
2,378,2,2013-01-01,0,0,0,0,a,1,a,...,8.0,2012.0,0.0,0.0,0.0,NoPromo,1,1,2013,1
3,377,2,2013-01-01,0,0,0,0,a,1,a,...,6.0,2010.0,1.0,18.0,2010.0,"Feb,May,Aug,Nov",1,1,2013,1
4,376,2,2013-01-01,0,0,0,0,a,1,a,...,8.0,2012.0,0.0,0.0,0.0,NoPromo,1,1,2013,1


## Eliminar Customers y Date

Aunque sea un buen indicador de las ventas, no podemos saber cuantos customers van a haber

In [26]:
df = df.drop(columns=["Customers", "Date"])

## Guardar en processed

Este es el dataframe final que se utilizará para Machine Learning

In [27]:
df.to_csv("../data/processed/df_processed.csv", index=False)